# Vocational NMT: run 5, terminology-aware distillation (experiment 2)

Compares, on the domain test sets, using the **formal** terminology glossary:
* `base`: IndicTrans2 dist-200M
* `lora`: run-4 adapter, distilled on raw teacher drafts (shipped in the bundle)
* `lora_formal`: **new** adapter, distilled on terminology-repaired drafts
* `teacher`: IndicTrans2 1B
* `…+cd`: the same model with glossary-constrained decoding

Steps:
0. Setup and smoke tests.
1. **Draft repair:** the 1B teacher re-translates non-compliant train/dev drafts with the glossary terms forced (2 GPUs).
2. **Fine-tuning** of `joint_formal` on GPU 0, while GPU 1 evaluates the systems that don't need it.
3. Evaluation of `lora_formal` (+cd), then merging and scoring of all systems.
4. IN22-Gen forgetting check for `lora_formal`.

**Input:** only the latest bundle dataset (`vocational-nmt-bundle.zip`). It already contains the raw teacher drafts and the run-4 adapter. Earlier run outputs aren't needed.

Use GPU T4 x2, Internet on, and the `HF_TOKEN` secret. Then **Save & Run All**; the outputs end up in `/kaggle/working/outputs.zip`.

In [ ]:
# 1. Copy the newest stamped bundle into a writable working directory (+ reuse earlier drafts)
import os, shutil, subprocess, sys, glob, zipfile
from pathlib import Path

PROJECT = Path("/kaggle/working/project")
if not PROJECT.exists():
    # Only folders/zips carrying kaggle/BUNDLE_VERSION count as code; outputs of earlier runs
    # (added as inputs to reuse drafts) also contain a backend/ folder with OLD code.
    candidates = []
    for z in glob.glob("/kaggle/input/**/vocational-nmt-bundle.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            if "kaggle/BUNDLE_VERSION" in zf.namelist():
                candidates.append((zf.read("kaggle/BUNDLE_VERSION").decode().strip(), z))
    for stamp in glob.glob("/kaggle/input/**/kaggle/BUNDLE_VERSION", recursive=True):
        candidates.append((Path(stamp).read_text().strip(), str(Path(stamp).parents[1])))
    if not candidates:
        raise FileNotFoundError("No bundle with kaggle/BUNDLE_VERSION under /kaggle/input - upload the latest bundle zip.")
    version, source = max(candidates)
    print(f"Using bundle {version} from {source}")
    if source.endswith(".zip"):
        zipfile.ZipFile(source).extractall(PROJECT)
    else:
        shutil.copytree(source, PROJECT)
print("bundle version:", (PROJECT / "kaggle/BUNDLE_VERSION").read_text().strip())
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(PROJECT)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation during beam search
sys.path.insert(0, str(PROJECT))

CORPUS = PROJECT / "backend/data/corpus_v2"
for pattern, sub in (("drafts_*.csv", ""), ("postedit/test_*.csv", "postedit")):
    for src in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        dst = CORPUS / sub / Path(src).name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src, dst)
            print("reused", src)
print(sorted(p.name for p in CORPUS.iterdir()))
assert (PROJECT / "backend/models/it2_lora_joint/adapter_model.safetensors").exists(), "run-4 adapter missing from bundle"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install the pinned stack (about 1 minute)
!pip install -q -r kaggle/requirements-kaggle.txt
!python -c "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"

In [ ]:
# 3. Hugging Face token + access check for the gated repos
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import hf_hub_download
IN22_OK = True
for repo, kind, fname in [("ai4bharat/indictrans2-en-indic-dist-200M", "model", "config.json"),
                          ("ai4bharat/indictrans2-en-indic-1B", "model", "config.json"),
                          ("ai4bharat/IN22-Gen", "dataset", "data/train-00000-of-00001.parquet")]:
    try:
        hf_hub_download(repo, fname, repo_type=kind, token=os.environ["HF_TOKEN"])
        print("access OK:", repo)
    except Exception as exc:
        print(f"NO ACCESS: {repo} -> accept its terms on huggingface.co ({type(exc).__name__})")
        if kind == "model":
            raise
        IN22_OK = False

In [ ]:
# 4. Smoke tests (~5 min): plain + constrained inference, and a 20-sentence constrained draft repair
def check(step):
    assert _exit_code == 0, f"{step} failed - see the output above"

!python -m backend.baseline_inference --text "Check the three phase supply and the earthing before switching on." --target-lang hin_Deva
check("base inference")
assert all((CORPUS / f"drafts_{s}.csv").exists() for s in ["hi", "gom", "mai", "doi"]), "raw drafts missing from the bundle"
!python -m backend.repair_drafts --langs hin_Deva --limit 20 --output-dir /kaggle/working/smoke_repair
check("draft repair smoke test")
import pandas as pd
smoke = pd.read_csv("/kaggle/working/smoke_repair/drafts_hi_formal.csv", keep_default_na=False)
smoke = smoke[smoke.term_repair != "kept"]
print(smoke.term_repair.value_counts().to_dict())
for _, r in smoke.head(5).iterrows():
    print(f"[{r.term_repair}] EN: {r.english_sentence[:90]}\n   NEW: {r.target_sentence[:90]}\n   {r.repair_note[:90]}")

## Step 1: terminology repair of the train/dev drafts
About 3,500–4,000 drafts per language need repair. The time depends on the constrained-search speed shown in the smoke test; the stats are written to `results/term_repair_stats_*.json`.

In [ ]:
import torch
n_gpu = torch.cuda.device_count()
groups = [["hin_Deva", "mai_Deva"], ["gom_Deva", "doi_Deva"]] if n_gpu >= 2 else [["hin_Deva", "mai_Deva", "gom_Deva", "doi_Deva"]]
procs = []
for gpu, langs in enumerate(groups):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"/kaggle/working/repair_gpu{gpu}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "-m", "backend.repair_drafts", "--langs", ",".join(langs)],
                                  env=env, stdout=log, stderr=subprocess.STDOUT))
codes = [p.wait() for p in procs]
for gpu in range(len(groups)):
    print(open(f"/kaggle/working/repair_gpu{gpu}.log").read()[-1500:])
assert all(c == 0 for c in codes), codes
!cat backend/results/term_repair_stats_*.json

In [ ]:
# Training smoke test (~3 min) on the repaired drafts, then load the resulting adapter
!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --drafts-variant formal --run-name smoke --max-steps 10 --eval-steps 5 --dev-per-lang 8
check("training smoke test")
!python -m backend.baseline_inference --model backend/models/it2_lora_smoke --text "Connect the neutral wire to the terminal block." --target-lang mai_Deva
check("adapter loading")
shutil.rmtree("backend/models/it2_lora_smoke", ignore_errors=True)

## Step 2: train `joint_formal` on GPU 0; evaluate base / lora / teacher (+cd) on GPU 1 meanwhile

In [ ]:
EVAL_A = "base,base+cd,lora,lora+cd,teacher"
eval_a = None
if n_gpu >= 2:
    eval_a = subprocess.Popen(
        [sys.executable, "-m", "backend.run_domain_eval", "--systems", EVAL_A, "--tag", "run5a"],
        env=dict(os.environ, CUDA_VISIBLE_DEVICES="1"),
        stdout=open("/kaggle/working/eval_run5a.log", "w"), stderr=subprocess.STDOUT)

!CUDA_VISIBLE_DEVICES=0 python -m backend.finetune --drafts-variant formal --run-name joint_formal
check("fine-tuning")

## Step 3: evaluate the new adapter, then score all systems together

In [ ]:
if eval_a is not None:
    print("eval A exit code:", eval_a.wait())
    print(open("/kaggle/working/eval_run5a.log").read()[-3000:])
    assert eval_a.returncode == 0
else:
    !CUDA_VISIBLE_DEVICES=0 python -m backend.run_domain_eval --systems {EVAL_A} --tag run5a
    check("evaluation A")
!CUDA_VISIBLE_DEVICES=0 python -m backend.run_domain_eval --systems lora_formal,lora_formal+cd --tag run5b
check("evaluation B")
!python -m backend.run_domain_eval --from-predictions backend/results/domain_predictions_UNVERIFIED_run5a.csv backend/results/domain_predictions_UNVERIFIED_run5b.csv --tag run5
check("merged scoring")

## Step 4: IN22-Gen forgetting check for `lora_formal`

In [ ]:
if IN22_OK:
    !CUDA_VISIBLE_DEVICES=0 python -m backend.run_baseline_eval --skip-base --adapter backend/models/it2_lora_joint_formal --output backend/results/in22_gen_lora_formal.csv
else:
    print("Skipped: no access to ai4bharat/IN22-Gen.")

In [ ]:
# Package everything that must come back to your machine
out = Path("/kaggle/working/outputs")
shutil.rmtree(out, ignore_errors=True)
(out / "corpus_v2").mkdir(parents=True)
for f in glob.glob("backend/data/corpus_v2/drafts_*_formal.csv"):
    shutil.copy(f, out / "corpus_v2")
for src, dst in [("backend/models/it2_lora_joint_formal", "it2_lora_joint_formal"), ("backend/results", "results")]:
    if Path(src).exists():
        shutil.copytree(src, out / dst, ignore=shutil.ignore_patterns("checkpoints"))
    else:
        print("missing:", src)
for log in glob.glob("/kaggle/working/*.log"):
    shutil.copy(log, out)
shutil.make_archive("/kaggle/working/outputs", "zip", out)
!du -sh /kaggle/working/outputs.zip